# Notebook 5 - Transaction Engine

this one took me a while to get right. the idea is simple - compare this quarter's holdings to last quarter's holdings and figure out what changed.

technically its a full outer join on cusip and put_call between consecutive quarters:
- position exists this quarter but not last quarter = NEW POSITION  
- position existed last quarter but not this quarter = FULL EXIT
- shares went up = BUY
- shares went down = SELL
- shares basically the same = HOLD

key thing - we compare SHARES not dollar values. dollar values change every quarter just from price movements even if the manager didnt do anything. shares only change when there is an actual transaction.

there is a small tolerance for rounding too, less than 0.5% share change we just call it HOLD.

In [1]:
import os, sys, pathlib

# ----- COLAB SETUP -----
# Option A: with Google Drive (data stays between sessions)
# from google.colab import drive
# drive.mount("/content/drive")
# PROJECT_ROOT = pathlib.Path("/content/drive/MyDrive/13F-Analytics")

# Option B: without Drive (data gone when session ends, need to rerun)
# PROJECT_ROOT = pathlib.Path("/content/13F-Analytics")

# detects root automatically when running locally
PROJECT_ROOT = pathlib.Path("/content/13F-Analytics") if pathlib.Path("/content/13F-Analytics").exists() else pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()

os.chdir(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("working directory:", PROJECT_ROOT)

working directory: /content/13F-Analytics


In [2]:
from src.transactions import build_transactions

tx = build_transactions()
tx.head(10)

,quarter,prev_quarter,cusip,put_call,issuer,title,asset_class,shares_prev,shares_curr,share_change,value_usd_prev,value_usd_curr,value_change_usd,portfolio_weight_prev,portfolio_weight_curr,action
0,2024Q3,2024Q2,02005N100,,ALLY FINL INC,COM,Common Stock,"29,000,000.00","29,000,000.00",0.00,"1,150,430,000.00","1,032,110,000.00","-118,320,000.00",0.00,0.00,HOLD
1,2024Q3,2024Q2,023135106,,AMAZON COM INC,COM,Common Stock,"10,000,000.00","10,000,000.00",0.00,"1,932,500,000.00","1,863,300,000.00","-69,200,000.00",0.01,0.01,HOLD
2,2024Q3,2024Q2,025816109,,AMERICAN EXPRESS CO,COM,Common Stock,"151,610,700.00","151,610,700.00",0.00,"35,105,457,585.00","41,116,821,840.00","6,011,364,255.00",0.13,0.15,HOLD
3,2024Q3,2024Q2,037833100,,APPLE INC,COM,Common Stock,"400,000,000.00","300,000,000.00","-100,000,000.00","84,248,000,000.00","69,900,000,000.00","-14,348,000,000.00",0.30,0.26,SELL
4,2024Q3,2024Q2,047726302,,ATLANTA BRAVES HLDGS INC,COM SER C,Common Stock,"223,645.00","223,645.00",0.00,"8,820,558.00","8,901,072.00","80,514.00",0.00,0.00,HOLD
5,2024Q3,2024Q2,060505104,,BANK AMER CORP,COM,Common Stock,"1,032,852,006.00","797,683,307.00","-235,168,699.00","41,076,524,279.00","31,652,073,622.00","-9,424,450,657.00",0.15,0.12,SELL
6,2024Q3,2024Q2,14040H105,,CAPITAL ONE FINL CORP,COM,Common Stock,"9,819,052.00","9,100,000.00","-719,052.00","1,359,447,750.00","1,362,543,000.00","3,095,250.00",0.00,0.01,SELL
7,2024Q3,2024Q2,16119P108,,CHARTER COMMUNICATIONS INC N,CL A,Common Stock,"3,828,941.00","2,821,879.00","-1,007,062.00","1,144,700,201.00","914,514,546.00","-230,185,655.00",0.00,0.00,SELL
8,2024Q3,2024Q2,166764100,,CHEVRON CORP NEW,COM,Common Stock,"118,610,534.00","118,610,534.00",0.00,"18,553,059,728.00","17,467,773,342.00","-1,085,286,386.00",0.07,0.07,HOLD
9,2024Q3,2024Q2,172967424,,CITIGROUP INC,COM NEW,Common Stock,"55,244,797.00","55,244,797.00",0.00,"3,505,834,818.00","3,458,324,292.00","-47,510,526.00",0.01,0.01,HOLD


In [3]:
# summary of actions per quarter
tx.pivot_table(index="quarter", columns="action", values="cusip", aggfunc="count", fill_value=0)

action,BUY,FULL EXIT,HOLD,NEW POSITION,SELL
quarter,,,,,
2024Q3,1,4,30,3,6
2024Q4,5,3,24,1,8
2025Q1,0,37,0,3,1
2025Q2,3,0,0,37,1
2025Q3,4,1,31,1,5
2025Q4,3,3,26,4,9
2026Q1,4,16,16,3,6


In [4]:
# biggest moves in the most recent quarter
latest = tx["quarter"].max()
(tx[(tx["quarter"] == latest) & (tx["action"] != "HOLD")]
   .assign(abs_change=lambda d: d["value_change_usd"].abs())
   .sort_values("abs_change", ascending=False)
   [["issuer", "cusip", "action", "share_change", "value_change_usd"]]
   .head(15))

,issuer,cusip,action,share_change,value_change_usd
256,ALPHABET INC,02079K305,BUY,"36,403,656.00","10,014,229,467.00"
261,BANK AMERICA CORP,060505104,SELL,"-3,671,769.00","-3,412,098,326.00"
293,VISA INC,92826C839,FULL EXIT,"-8,297,460.00","-2,910,002,197.00"
268,DELTA AIR LINES INC,247361702,NEW POSITION,"39,809,456.00","2,646,532,635.00"
264,CHEVRON CORPORATION,166764100,SELL,"-45,780,506.00","-2,379,766,525.00"
283,MASTERCARD INCORPORATED,57636Q104,FULL EXIT,"-3,986,648.00","-2,275,897,610.00"
266,CONSTELLATION BRANDS INC,21036P108,SELL,"-12,367,110.00","-1,698,546,500.00"
291,UNITEDHEALTH GROUP INC,91324P102,FULL EXIT,"-5,039,564.00","-1,663,610,472.00"
270,DOMINOS PIZZA INC,25754A201,FULL EXIT,"-3,350,000.00","-1,396,347,000.00"
295,AON PLC,G0403H108,FULL EXIT,"-3,602,995.00","-1,271,424,876.00"
